# Isolation Forest for Anomaly Detection

## What This Notebook Covers
This notebook implements **Isolation Forest** for anomaly detection — both from scratch using NumPy and using scikit-learn's `IsolationForest` class. Isolation Forest isolates anomalies by random recursive partitioning: anomalies (few and different) are isolated in fewer splits than normal points.

## Prerequisites
- Binary tree data structures (recursive partitioning)
- Logarithms and harmonic numbers
- Python, NumPy, pandas, matplotlib

## Dataset
**Credit Card Fraud Detection** (Kaggle)  
Link: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud  
284,807 transactions with 30 features (V1-V28 from PCA + Time + Amount). Target: `Class` (0=normal, 1=fraud, ~0.17% fraud rate).

## Credits
- Dataset by ULB Machine Learning Group
- Liu, Ting, Zhou (2008) — Original Isolation Forest paper
- Gradientts ML Intern Program — 2026

In [ ]:
# ============================================================
# Cell 2: All Imports
# ============================================================

import numpy as np                   # Core numerical computing
import pandas as pd                  # Data manipulation and loading
import matplotlib.pyplot as plt      # Plotting and visualisation
import seaborn as sns                # Statistical visualisation
from sklearn.preprocessing import StandardScaler  # Feature scaling
from sklearn.model_selection import train_test_split  # Splitting
from sklearn.ensemble import IsolationForest  # Sklearn Isolation Forest
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    roc_curve
)

# Reproducibility
np.random.seed(42)

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All imports successful.')

## Part 1: Theory Recap

**Isolation Forest** detects anomalies by measuring how easily a point can be isolated via random partitioning:

1. **Core principle:** Anomalies are few and different — they are isolated in fewer random splits than normal points.
2. **Isolation Tree:** Recursively pick a random feature and a random split value. Anomalies reach leaf nodes with short path lengths.
3. **Anomaly Score:** $s(x, n) = 2^{-E[h(x)] / c(n)}$, where $E[h(x)]$ is the average path length and $c(n)$ is the normalisation constant.
4. **Score interpretation:** $s \to 1$ = anomaly (short paths), $s \approx 0.5$ = normal, $s \to 0$ = very normal (long paths).
5. **Key advantage:** Linear time complexity O(n·t·log ψ), handles high-dimensional data well, no density estimation needed.

## Loading the Dataset

We load the Credit Card Fraud Detection dataset. Isolation Forest scales well, so we can use more data than KDE.

In [ ]:
# ============================================================
# Cell 4: Load the real-world dataset
# ============================================================

try:
    df = pd.read_csv('creditcard.csv')
    print('Loaded from local file.')
except FileNotFoundError:
    from sklearn.datasets import fetch_openml
    print('Local file not found. Downloading from OpenML...')
    credit = fetch_openml(data_id=1597, as_frame=True, parser='auto')
    df = credit.frame
    if 'Class' not in df.columns:
        df = df.rename(columns={df.columns[-1]: 'Class'})
    df['Class'] = df['Class'].astype(int)
    print('Downloaded successfully.')

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:\n{df["Class"].value_counts()}')
print(f'\nFraud rate: {df["Class"].mean()*100:.3f}%')

print('\n--- First 5 rows ---')
df.head()

In [ ]:
print('--- Dataset Info ---')
print(df.info())
print('\n--- Statistical Summary ---')
df.describe()

## Preprocessing

Scale `Time` and `Amount`, create a subsample for the from-scratch implementation (the scratch version is slower than sklearn's optimised C code).

In [ ]:
# ============================================================
# Cell 5: Preprocessing
# ============================================================

# Step 1: Scale Time and Amount
scaler = StandardScaler()
df['scaled_Amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_Time'] = scaler.fit_transform(df[['Time']])
df_clean = df.drop(['Time', 'Amount'], axis=1)

# Step 2: Check for nulls
print(f'Null values: {df_clean.isnull().sum().sum()}')

# Step 3: Create subsample for scratch implementation
fraud = df_clean[df_clean['Class'] == 1]
normal = df_clean[df_clean['Class'] == 0].sample(n=5000, random_state=42)
df_sub = pd.concat([normal, fraud]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Subsample shape: {df_sub.shape}')
print(f'Class distribution:\n{df_sub["Class"].value_counts()}')

# Step 4: Separate features and labels
X_sub = df_sub.drop('Class', axis=1).values
y_sub = df_sub['Class'].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_sub, y_sub, test_size=0.3, random_state=42, stratify=y_sub
)

print(f'\nTraining set: {X_train.shape}')
print(f'Test set: {X_test.shape} (fraud: {y_test.sum()}, normal: {(y_test==0).sum()})')

## Part 2: From Scratch Implementation

We implement Isolation Forest from scratch using only NumPy.

**What we are building:**
- `IsolationTreeScratch` — a single random tree that recursively partitions data with random feature + random split value
- `IsolationForestScratch` — an ensemble of isolation trees that averages path lengths to compute anomaly scores

**Why from scratch?** Understanding the tree construction, path length computation, and the c(n) normalisation is essential for explaining how IF avoids the curse of dimensionality (it doesn't compute distances or densities).

In [ ]:
# ============================================================
# Cell 7: From-Scratch Isolation Forest (NumPy only)
# ============================================================

def _c(n):
    """
    Average path length of unsuccessful search in BST.
    c(n) = 2*H(n-1) - 2*(n-1)/n
    where H(k) = ln(k) + Euler-Mascheroni constant (0.5772...)
    """
    # INTERVIEW NOTE: This normalisation factor is crucial.
    # It adjusts for the fact that trees with more data points
    # naturally produce longer paths.
    if n <= 1:
        return 0
    if n == 2:
        return 1
    # H(n-1) ≈ ln(n-1) + 0.5772156649 (Euler-Mascheroni)
    H = np.log(n - 1) + 0.5772156649
    return 2.0 * H - 2.0 * (n - 1) / n


class IsolationTreeScratch:
    """
    A single Isolation Tree.
    Recursively partitions data using random feature + random split value.
    Anomalies are isolated quickly (short path to leaf).
    """
    
    def __init__(self, max_depth):
        self.max_depth = max_depth
        self.is_leaf = False
        self.size = 0
        self.feature = None
        self.threshold = None
        self.left = None
        self.right = None
    
    def fit(self, X, depth=0):
        n, d = X.shape
        
        # INTERVIEW NOTE: Stop conditions create leaf nodes.
        # Leaf size > 1 means the remaining points are adjusted with c(size).
        if depth >= self.max_depth or n <= 1:
            self.is_leaf = True
            self.size = n
            return self
        
        # INTERVIEW NOTE: Random feature + random split = key innovation.
        # No information gain, no criterion — pure randomness.
        # This is what makes anomalies isolatable in few splits.
        self.feature = np.random.randint(d)
        feat_values = X[:, self.feature]
        min_val, max_val = feat_values.min(), feat_values.max()
        
        # If all values are the same, can't split
        if min_val == max_val:
            self.is_leaf = True
            self.size = n
            return self
        
        self.threshold = np.random.uniform(min_val, max_val)
        
        # Partition data
        left_mask = feat_values < self.threshold
        self.left = IsolationTreeScratch(self.max_depth).fit(X[left_mask], depth + 1)
        self.right = IsolationTreeScratch(self.max_depth).fit(X[~left_mask], depth + 1)
        
        return self
    
    def path_length(self, x, depth=0):
        """
        Compute the path length for a single data point.
        Returns depth + c(leaf_size) adjustment.
        """
        # INTERVIEW NOTE: The c(size) adjustment at leaves accounts for
        # the expected additional path length if we had continued splitting.
        if self.is_leaf:
            return depth + _c(self.size)
        
        if x[self.feature] < self.threshold:
            return self.left.path_length(x, depth + 1)
        else:
            return self.right.path_length(x, depth + 1)


class IsolationForestScratch:
    """
    Isolation Forest ensemble for anomaly detection.
    Builds multiple isolation trees on subsamples and
    averages path lengths to compute anomaly scores.
    """
    
    def __init__(self, n_estimators=100, max_samples=256):
        # INTERVIEW NOTE: max_samples=256 is the original paper's recommendation.
        # Small subsamples help: (1) speed, (2) reduce swamping/masking effects.
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.trees = []
    
    def fit(self, X):
        n = len(X)
        subsample_size = min(self.max_samples, n)
        max_depth = int(np.ceil(np.log2(subsample_size)))
        
        # INTERVIEW NOTE: Each tree is built on a DIFFERENT random subsample.
        # This is bagging — reduces variance and increases diversity.
        self.trees = []
        for _ in range(self.n_estimators):
            idx = np.random.choice(n, subsample_size, replace=False)
            tree = IsolationTreeScratch(max_depth).fit(X[idx])
            self.trees.append(tree)
        
        self._c_n = _c(subsample_size)  # Store for scoring
        return self
    
    def _anomaly_score(self, x):
        """
        Compute anomaly score for a single point.
        s(x,n) = 2^(-E[h(x)] / c(n))
        """
        # Average path length across all trees
        avg_path = np.mean([tree.path_length(x) for tree in self.trees])
        # Anomaly score formula from the original paper
        score = 2.0 ** (-avg_path / self._c_n)
        return score
    
    def score_samples(self, X):
        """
        Compute anomaly scores for all points.
        Returns NEGATIVE scores (for consistency with sklearn:
        more negative = more anomalous).
        """
        scores = np.array([self._anomaly_score(x) for x in X])
        return -scores  # Negate: sklearn convention
    
    def predict(self, X, contamination=0.01):
        """
        Predict labels: -1 = anomaly, 1 = normal.
        """
        scores = self.score_samples(X)
        threshold = np.percentile(scores, contamination * 100)
        return np.where(scores <= threshold, -1, 1)


print('IsolationForestScratch class defined successfully.')
print(f'c(256) = {_c(256):.4f} (normalisation factor for 256 subsamples)')

## Fitting and Evaluating the From-Scratch Isolation Forest

We fit on the full training set (including both normal and fraud — Isolation Forest is unsupervised and doesn't need clean training data). Then evaluate on the test set.

In [ ]:
# ============================================================
# Cell 8: Fit scratch model and evaluate
# ============================================================

# Fit the scratch Isolation Forest
print('Building Isolation Forest from scratch (100 trees, 256 subsamples)...')
if_scratch = IsolationForestScratch(n_estimators=100, max_samples=256)
if_scratch.fit(X_train)

# Score the test set
print('Scoring test set...')
scratch_scores = if_scratch.score_samples(X_test)

# Predict with contamination matching fraud rate
contamination = y_test.mean()
scratch_preds = if_scratch.predict(X_test, contamination=contamination)
scratch_labels = np.where(scratch_preds == -1, 1, 0)

print('\n--- From-Scratch Isolation Forest Results ---')
print(classification_report(y_test, scratch_labels, target_names=['Normal', 'Fraud']))

# AUC using raw scores (more negative = more anomalous)
scratch_auc = roc_auc_score(y_test, -scratch_scores)  # Negate back to positive = anomalous
print(f'AUC-ROC: {scratch_auc:.4f}')

scratch_ap = average_precision_score(y_test, -scratch_scores)
print(f'Average Precision (PR-AUC): {scratch_ap:.4f}')

## Part 3: Sklearn Implementation

Scikit-learn's `IsolationForest` is implemented in optimised C/Cython code and provides:
- Automatic `contamination` parameter for thresholding
- `decision_function()` which returns the negative anomaly score
- Built-in `predict()` returning -1 (anomaly) or 1 (normal)

Key difference from our scratch version: sklearn uses the full feature set efficiently and provides warm_start, n_jobs parallelism.

In [ ]:
# ============================================================
# Cell 10: Sklearn Isolation Forest
# ============================================================

# Fit sklearn Isolation Forest
sklearn_if = IsolationForest(
    n_estimators=100,
    max_samples=256,
    contamination=float(contamination),
    random_state=42
)
sklearn_if.fit(X_train)

# Predict and score
sklearn_preds = sklearn_if.predict(X_test)
sklearn_labels = np.where(sklearn_preds == -1, 1, 0)
sklearn_scores = sklearn_if.decision_function(X_test)  # Higher = more normal

print('--- Sklearn Isolation Forest Results ---')
print(classification_report(y_test, sklearn_labels, target_names=['Normal', 'Fraud']))

sklearn_auc = roc_auc_score(y_test, -sklearn_scores)  # Negate: lower decision = anomaly
print(f'AUC-ROC: {sklearn_auc:.4f}')

sklearn_ap = average_precision_score(y_test, -sklearn_scores)
print(f'Average Precision (PR-AUC): {sklearn_ap:.4f}')

# --- Direct Comparison ---
print('\n=== Scratch vs Sklearn Comparison ===')
print(f'{"Metric":<25} {"Scratch":<15} {"Sklearn":<15}')
print(f'{"AUC-ROC":<25} {scratch_auc:<15.4f} {sklearn_auc:<15.4f}')
print(f'{"Avg Precision (PR-AUC)":<25} {scratch_ap:<15.4f} {sklearn_ap:<15.4f}')

## Visualisations

1. **Anomaly Score Distribution** — normal vs fraud scores
2. **ROC Curve** — scratch vs sklearn comparison

In [ ]:
# ============================================================
# Cell 11: Visualisations
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Anomaly Score Distribution ---
ax1 = axes[0]
ax1.hist(-sklearn_scores[y_test == 0], bins=50, alpha=0.7, label='Normal', color='steelblue', density=True)
ax1.hist(-sklearn_scores[y_test == 1], bins=50, alpha=0.7, label='Fraud', color='crimson', density=True)
ax1.set_xlabel('Anomaly Score (higher = more anomalous)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Isolation Forest: Anomaly Score Distribution', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)

# --- Plot 2: ROC Curve ---
ax2 = axes[1]
fpr_s, tpr_s, _ = roc_curve(y_test, -scratch_scores)
fpr_sk, tpr_sk, _ = roc_curve(y_test, -sklearn_scores)
ax2.plot(fpr_s, tpr_s, label=f'Scratch (AUC={scratch_auc:.3f})', color='darkorange', linewidth=2)
ax2.plot(fpr_sk, tpr_sk, label=f'Sklearn (AUC={sklearn_auc:.3f})', color='steelblue', linewidth=2)
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title('ROC Curve: Scratch vs Sklearn IF', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

# --- Plot 3: Precision-Recall Curve ---
fig, ax = plt.subplots(figsize=(8, 6))
prec, rec, _ = precision_recall_curve(y_test, -sklearn_scores)
ax.plot(rec, prec, color='steelblue', linewidth=2, label=f'IF (AP={sklearn_ap:.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve (Isolation Forest)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Part 4: Hyperparameter Experiments

Key hyperparameters for Isolation Forest:

1. **`n_estimators`** — number of trees. More trees = more stable scores but slower.
2. **`max_samples`** — subsample size per tree. Controls the trade-off between swamping/masking and tree diversity.

We vary both and measure AUC-ROC.

In [ ]:
# ============================================================
# Cell 13: Hyperparameter Experiments
# ============================================================

# --- Experiment 1: n_estimators effect ---
n_estimators_range = [10, 25, 50, 100, 200, 300, 500]
ne_aucs = []

for ne in n_estimators_range:
    model = IsolationForest(n_estimators=ne, max_samples=256,
                            contamination=float(contamination), random_state=42)
    model.fit(X_train)
    scores = model.decision_function(X_test)
    auc_val = roc_auc_score(y_test, -scores)
    ne_aucs.append(auc_val)
    print(f'n_estimators={ne:<5d}  AUC-ROC={auc_val:.4f}')

# --- Experiment 2: max_samples effect ---
max_samples_range = [32, 64, 128, 256, 512, 1024, 2048]
ms_aucs = []

for ms in max_samples_range:
    if ms > len(X_train):
        continue
    model = IsolationForest(n_estimators=100, max_samples=ms,
                            contamination=float(contamination), random_state=42)
    model.fit(X_train)
    scores = model.decision_function(X_test)
    auc_val = roc_auc_score(y_test, -scores)
    ms_aucs.append(auc_val)
    print(f'max_samples={ms:<6d}  AUC-ROC={auc_val:.4f}')

# --- Plot results ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(n_estimators_range, ne_aucs, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Estimators', fontsize=12)
axes[0].set_ylabel('AUC-ROC', fontsize=12)
axes[0].set_title('Effect of n_estimators on IF Performance', fontsize=13, fontweight='bold')

valid_ms = [ms for ms in max_samples_range if ms <= len(X_train)]
axes[1].plot(valid_ms, ms_aucs, 's-', color='coral', linewidth=2, markersize=8)
axes[1].set_xlabel('Max Samples (ψ)', fontsize=12)
axes[1].set_ylabel('AUC-ROC', fontsize=12)
axes[1].set_title('Effect of max_samples on IF Performance', fontsize=13, fontweight='bold')
axes[1].set_xscale('log', base=2)

plt.tight_layout()
plt.show()

## Part 5: Interview Corner

### Key Conceptual Question: *"How does Isolation Forest avoid the curse of dimensionality that plagues density-based methods?"*

**Answer Structure:**

1. **No density estimation:** IF doesn't compute distances or density. It only asks "how many random splits to isolate this point?" — a fundamentally different question from "how dense is this region?"

2. **Random feature selection per split:** Each split uses ONE random feature. In high dimensions, anomalies typically have extreme values on at least a few features. Random selection eventually picks those features and isolates the anomaly.

3. **Sublinear depth:** Tree depth is O(log ψ), independent of dimensionality d. A point with an extreme value on ANY feature gets isolated quickly, regardless of how many total features exist.

4. **Contrast with KDE/LOF:** KDE uses all d dimensions simultaneously (density → 0 in high-d). LOF uses k-NN distances (distances converge in high-d). IF's single-feature splits sidestep both issues.

5. **Caveat:** If anomalies only differ in feature *combinations* (not individual extreme values), IF may still struggle. Extended Isolation Forest (EIF) addresses this by using random hyperplanes instead of axis-aligned splits.

## Key Takeaways

1. **Isolation, not profiling** — Isolation Forest directly targets anomalies by measuring isolation difficulty, unlike density methods that first model normal behaviour and then flag deviations.

2. **Linear time complexity** — O(n·t·log ψ) makes IF one of the fastest anomaly detectors, suitable for datasets with millions of rows.

3. **The anomaly score formula s(x,n) = 2^(-E[h(x)]/c(n))** normalises the average path length. Understanding c(n) and its derivation from BST theory is a common interview topic.

4. **max_samples = 256 is surprisingly effective** — the original paper showed that small subsamples actually *improve* performance by reducing swamping (anomalies diluted by normal data) and masking (anomalies hidden by other anomalies).

5. **Handles high dimensions gracefully** — since each split uses only one feature, IF avoids the curse of dimensionality that cripples KDE and LOF in 20+ dimensional spaces.